# Data Structures for AI and Data Engineering

Lists, dicts, sets, tuples, dataclasses, and the list-of-dicts pattern that represents 90% of data in Python.

**Concept chapter:** [Data Structures](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/python/02_Data_Structures.md)

**Community:** [Delivery Momentum on Skool](https://www.skool.com/deliverymomentum)

## 1. Lists — Ordered, Mutable Collections

Lists are Python's workhorse. Feature vectors, batch IDs, column names — all lists.

In [ ]:
# WHAT: Create, index, slice, and modify lists.
# WHY: Feature columns, batch IDs, and pipeline stages are all lists.

feature_columns = ["age", "income", "credit_score", "tenure_months"]

# Indexing — zero-based, negative indexes count from the end
print(f"First feature:  {feature_columns[0]}")
print(f"Last feature:   {feature_columns[-1]}")

# Slicing — same [start:stop] as strings
print(f"First two:      {feature_columns[:2]}")

# Modify in place
feature_columns.append("debt_ratio")
print(f"After append:   {feature_columns}")

# Extend with another list
new_features = ["employment_type", "region"]
feature_columns.extend(new_features)
print(f"After extend:   {feature_columns}")
print(f"Total features: {len(feature_columns)}")

# You Should See:
# First feature:  age
# Last feature:   tenure_months
# First two:      ['age', 'income']
# After append:   ['age', 'income', 'credit_score', 'tenure_months', 'debt_ratio']
# After extend:   ['age', 'income', 'credit_score', 'tenure_months', 'debt_ratio', 'employment_type', 'region']
# Total features: 7

In [ ]:
# WHAT: Sort and reverse lists.
# WHY: Ranking predictions, sorting metrics for reporting.

scores = [0.72, 0.95, 0.88, 0.41, 0.63]

# sorted() returns a NEW list — original is unchanged
ranked = sorted(scores, reverse=True)
print(f"Original:       {scores}")
print(f"Ranked (desc):  {ranked}")

# .sort() modifies IN PLACE — returns None
# Common mistake: result = scores.sort() gives you None
scores.sort()
print(f"After .sort():  {scores}")

# You Should See:
# Original:       [0.72, 0.95, 0.88, 0.41, 0.63]
# Ranked (desc):  [0.95, 0.88, 0.72, 0.63, 0.41]
# After .sort():  [0.41, 0.63, 0.72, 0.88, 0.95]

## 2. List Comprehensions

The Pythonic way to transform and filter data. Replaces `for` + `append` patterns.

In [ ]:
# WHAT: Transform data with list comprehensions.
# WHY: Cleaner than a for-loop with append. You will see this
# in every Python codebase.

raw_scores = [0.72, 0.95, 0.88, 0.41, 0.63]

# Basic: transform every element
percentages = [round(s * 100, 1) for s in raw_scores]
print(f"Percentages: {percentages}")

# Filtered: only elements meeting a condition
high_confidence = [s for s in raw_scores if s >= 0.70]
print(f"High confidence: {high_confidence}")

# Transform + filter: normalize only passing scores
labels = [f"PASS ({s:.0%})" if s >= 0.5 else f"FAIL ({s:.0%})" for s in raw_scores]
print(f"Labels: {labels}")

# Nested: flatten a list of feature groups
feature_groups = [["age", "income"], ["clicks", "views"], ["region"]]
flat = [feat for group in feature_groups for feat in group]
print(f"Flattened: {flat}")

# You Should See:
# Percentages: [72.0, 95.0, 88.0, 41.0, 63.0]
# High confidence: [0.72, 0.95, 0.88]
# Labels: ['PASS (72%)', 'PASS (95%)', 'PASS (88%)', 'FAIL (41%)', 'PASS (63%)']
# Flattened: ['age', 'income', 'clicks', 'views', 'region']

## 3. Dictionaries — Key-Value Lookup

Dicts are Python's hash maps. Config objects, JSON payloads, feature stores — all dicts.

In [ ]:
# WHAT: Create and access dictionaries.
# WHY: Every config file, API response, and feature store row
# is a dictionary in Python.

model_config = {
    "model_name": "xgboost-churn-v2",
    "learning_rate": 0.01,
    "max_depth": 6,
    "n_estimators": 500,
    "target_column": "churned"
}

# Access by key
print(f"Model: {model_config['model_name']}")
print(f"LR:    {model_config['learning_rate']}")

# .get() returns None instead of raising KeyError
# WHY: Defensive access when keys may be missing
gpu = model_config.get("gpu_enabled", False)
print(f"GPU:   {gpu}  (default because key missing)")

# Common mistake: model_config["gpu_enabled"] raises KeyError

# You Should See:
# Model: xgboost-churn-v2
# LR:    0.01
# GPU:   False  (default because key missing)

In [ ]:
# WHAT: Iterate over dicts with .items(), .keys(), .values().
# WHY: Logging configs, validating parameters, building reports.

model_config = {
    "model_name": "xgboost-churn-v2",
    "learning_rate": 0.01,
    "max_depth": 6,
    "n_estimators": 500
}

# .items() gives key-value pairs
print("Model configuration:")
for key, value in model_config.items():
    print(f"  {key:>15}: {value}")

# Check if a key exists
print(f"\n'max_depth' in config: {'max_depth' in model_config}")
print(f"'dropout' in config:   {'dropout' in model_config}")

# You Should See:
# Model configuration:
#      model_name: xgboost-churn-v2
#   learning_rate: 0.01
#       max_depth: 6
#    n_estimators: 500
# 'max_depth' in config: True
# 'dropout' in config:   False

In [ ]:
# WHAT: Nested dictionaries for structured configuration.
# WHY: Real configs are nested — pipeline > stage > parameters.

pipeline_config = {
    "extract": {
        "source": "s3://data-lake/raw/calls/",
        "format": "parquet"
    },
    "transform": {
        "dedup_column": "call_id",
        "null_threshold": 0.05
    },
    "load": {
        "target": "warehouse.silver.calls",
        "mode": "upsert"
    }
}

# Chain access for nested keys
source = pipeline_config["extract"]["source"]
target = pipeline_config["load"]["target"]
print(f"Source: {source}")
print(f"Target: {target}")

# You Should See:
# Source: s3://data-lake/raw/calls/
# Target: warehouse.silver.calls

In [ ]:
# WHAT: Dict comprehension to transform or filter dicts.
# WHY: Rename columns, filter configs, build lookup tables.

# Rename columns: snake_case to Title Case for a report
column_map = {"call_id": "Call ID", "duration_sec": "Duration (s)", "agent_name": "Agent"}
reversed_map = {v: k for k, v in column_map.items()}
print(f"Reversed: {reversed_map}")

# Filter to only numeric hyperparameters
config = {"model": "xgboost", "lr": 0.01, "depth": 6, "target": "churned"}
numeric_params = {k: v for k, v in config.items() if isinstance(v, (int, float))}
print(f"Numeric only: {numeric_params}")

# You Should See:
# Reversed: {'Call ID': 'call_id', 'Duration (s)': 'duration_sec', 'Agent': 'agent_name'}
# Numeric only: {'lr': 0.01, 'depth': 6}

## 4. Sets — Unique, Unordered Collections

Sets are for deduplication and membership testing. Checking `x in set` is O(1), vs O(n) for lists.

In [ ]:
# WHAT: Dedup and compare sets of IDs.
# WHY: Finding duplicates, missing records, and overlap between
# datasets is a daily DE task.

# Raw call IDs with duplicates
raw_ids = ["C001", "C002", "C003", "C002", "C004", "C001", "C005"]
unique_ids = set(raw_ids)
print(f"Raw count:    {len(raw_ids)}")
print(f"Unique count: {len(unique_ids)}")
print(f"Duplicates:   {len(raw_ids) - len(unique_ids)}")

# Set operations — compare two data sources
source_a = {"C001", "C002", "C003", "C004"}
source_b = {"C003", "C004", "C005", "C006"}

print(f"\nIn both:      {source_a & source_b}")        # intersection
print(f"In A only:    {source_a - source_b}")          # difference
print(f"In either:    {source_a | source_b}")          # union
print(f"In one only:  {source_a ^ source_b}")          # symmetric difference

# You Should See:
# Raw count:    7
# Unique count: 5
# Duplicates:   2
# In both:      {'C003', 'C004'}
# In A only:    {'C001', 'C002'}
# In either:    {'C001', 'C002', 'C003', 'C004', 'C005', 'C006'}
# In one only:  {'C001', 'C002', 'C005', 'C006'}

## 5. Tuples — Immutable Sequences

Tuples are fixed-size, immutable lists. Use them for coordinates, return values, and dict keys.

In [ ]:
# WHAT: Create tuples and unpack them.
# WHY: Functions return multiple values as tuples.
# NumPy's .shape is a tuple. Pandas index can be tuples.

# Tuple creation and unpacking
model_shape = (128, 64, 32)   # layer dimensions
input_dim, hidden_dim, output_dim = model_shape   # unpack
print(f"Input: {input_dim}, Hidden: {hidden_dim}, Output: {output_dim}")

# Tuples as dict keys — valid because they're immutable
# WHY: Cache lookup by (model_name, version) pair
model_cache = {
    ("bert", "v1"): 0.89,
    ("bert", "v2"): 0.92,
    ("gpt", "v1"): 0.91
}
print(f"bert-v2 accuracy: {model_cache[('bert', 'v2')]}")

# Common mistake: lists can NOT be dict keys (they're mutable)
# model_cache[["bert", "v1"]] would raise TypeError

# You Should See:
# Input: 128, Hidden: 64, Output: 32
# bert-v2 accuracy: 0.92

In [ ]:
# WHAT: Named tuples for self-documenting data.
# WHY: Better than plain tuples when fields have meaning.
# Lightweight alternative to a full class.

from collections import namedtuple

ModelResult = namedtuple("ModelResult", ["name", "accuracy", "latency_ms"])

result = ModelResult(name="xgboost-v2", accuracy=0.94, latency_ms=12)

# Access by name (readable) or index (compatible with tuples)
print(f"Model:    {result.name}")
print(f"Accuracy: {result.accuracy}")
print(f"Latency:  {result.latency_ms}ms")
print(f"As tuple: {tuple(result)}")

# You Should See:
# Model:    xgboost-v2
# Accuracy: 0.94
# Latency:  12ms
# As tuple: ('xgboost-v2', 0.94, 12)

## 6. Dataclasses — Clean Data Containers

Dataclasses auto-generate `__init__`, `__repr__`, and `__eq__`. They are the modern way to define structured data in Python.

In [ ]:
# WHAT: Define a clean data container with @dataclass.
# WHY: Replaces verbose __init__ boilerplate. Used in FastAPI,
# MLflow tracking, and config management.

from dataclasses import dataclass

@dataclass
class PipelineRun:
    pipeline_name: str
    rows_processed: int
    duration_sec: float
    status: str = "pending"   # default value

# Create instances — no boilerplate __init__ needed
run1 = PipelineRun("daily_etl", 1_250_000, 47.3, "success")
run2 = PipelineRun("hourly_agg", 50_000, 3.1)

print(run1)   # auto-generated __repr__
print(run2)
print(f"\nRun1 == Run2: {run1 == run2}")   # auto-generated __eq__

# You Should See:
# PipelineRun(pipeline_name='daily_etl', rows_processed=1250000, duration_sec=47.3, status='success')
# PipelineRun(pipeline_name='hourly_agg', rows_processed=50000, duration_sec=3.1, status='pending')
# Run1 == Run2: False

## 7. List of Dicts — How Data Actually Looks

This is THE pattern. JSON responses, database rows, CSV readers, and Pandas DataFrames all produce or consume lists of dicts.

In [ ]:
# WHAT: Work with a list of dicts — the universal data format.
# WHY: API responses, database cursors, and json.load() all return
# this structure. Master this and you can handle any data source.

predictions = [
    {"id": "P001", "model": "bert-v2", "score": 0.95, "label": "positive"},
    {"id": "P002", "model": "bert-v2", "score": 0.42, "label": "negative"},
    {"id": "P003", "model": "bert-v1", "score": 0.88, "label": "positive"},
    {"id": "P004", "model": "bert-v1", "score": 0.31, "label": "negative"},
    {"id": "P005", "model": "bert-v2", "score": 0.77, "label": "positive"}
]

# Iterate and print
print("All predictions:")
for p in predictions:
    print(f"  {p['id']}: {p['score']:.2f} ({p['label']})")

# You Should See:
# All predictions:
#   P001: 0.95 (positive)
#   P002: 0.42 (negative)
#   P003: 0.88 (positive)
#   P004: 0.31 (negative)
#   P005: 0.77 (positive)

In [ ]:
# WHAT: Filter and transform a list of dicts.
# WHY: This is what you do before loading data into a DataFrame
# or writing to a database.

predictions = [
    {"id": "P001", "model": "bert-v2", "score": 0.95, "label": "positive"},
    {"id": "P002", "model": "bert-v2", "score": 0.42, "label": "negative"},
    {"id": "P003", "model": "bert-v1", "score": 0.88, "label": "positive"},
    {"id": "P004", "model": "bert-v1", "score": 0.31, "label": "negative"},
    {"id": "P005", "model": "bert-v2", "score": 0.77, "label": "positive"}
]

# Filter: high confidence only
high_conf = [p for p in predictions if p["score"] >= 0.75]
print(f"High confidence ({len(high_conf)}):")
for p in high_conf:
    print(f"  {p['id']}: {p['score']}")

# Transform: extract just IDs and scores
summary = [{"id": p["id"], "score": p["score"]} for p in predictions]
print(f"\nSummary: {summary}")

# Aggregate: average score by model
from collections import defaultdict
scores_by_model = defaultdict(list)
for p in predictions:
    scores_by_model[p["model"]].append(p["score"])

print("\nAverage score by model:")
for model, scores in scores_by_model.items():
    avg = sum(scores) / len(scores)
    print(f"  {model}: {avg:.3f}")

# You Should See:
# High confidence (3):
#   P001: 0.95
#   P003: 0.88
#   P005: 0.77
# Summary: [{'id': 'P001', 'score': 0.95}, ...]
# Average score by model:
#   bert-v2: 0.713
#   bert-v1: 0.595

In [ ]:
# WHAT: Sort a list of dicts by a field.
# WHY: Ranking results, finding top-N, ordering for reports.

predictions = [
    {"id": "P001", "score": 0.95},
    {"id": "P002", "score": 0.42},
    {"id": "P003", "score": 0.88},
    {"id": "P004", "score": 0.31},
    {"id": "P005", "score": 0.77}
]

# Sort by score descending — key= takes a function
ranked = sorted(predictions, key=lambda p: p["score"], reverse=True)

print("Top 3 predictions:")
for i, p in enumerate(ranked[:3], 1):
    print(f"  #{i}: {p['id']} ({p['score']:.2f})")

# You Should See:
# Top 3 predictions:
#   #1: P001 (0.95)
#   #2: P003 (0.88)
#   #3: P005 (0.77)

In [ ]:
# WHAT: Build a lookup dict from a list of dicts.
# WHY: O(1) lookups by ID instead of scanning the whole list.
# This is how you build in-memory indexes.

predictions = [
    {"id": "P001", "score": 0.95, "label": "positive"},
    {"id": "P002", "score": 0.42, "label": "negative"},
    {"id": "P003", "score": 0.88, "label": "positive"}
]

# Dict comprehension to build index
by_id = {p["id"]: p for p in predictions}

# Now O(1) lookup
result = by_id["P002"]
print(f"Lookup P002: {result}")

# Safe lookup with .get()
missing = by_id.get("P999", {"id": "P999", "score": None, "label": "unknown"})
print(f"Lookup P999: {missing}")

# You Should See:
# Lookup P002: {'id': 'P002', 'score': 0.42, 'label': 'negative'}
# Lookup P999: {'id': 'P999', 'score': None, 'label': 'unknown'}

## 8. Choosing the Right Structure

| Need | Use | Why |
|---|---|---|
| Ordered collection, may change | `list` | Append, slice, iterate |
| Key-value lookup | `dict` | O(1) access by key |
| Unique values, membership test | `set` | O(1) `in` check, dedup |
| Fixed-size, immutable | `tuple` | Dict keys, return values |
| Structured data with fields | `dataclass` | Clean, typed, auto-repr |
| Tabular data from JSON/API | `list[dict]` | Universal exchange format |

In [ ]:
# WHAT: Performance comparison — set vs list for membership testing.
# WHY: Choosing the wrong structure turns O(1) into O(n).
# On 1M records, that's the difference between 1ms and 10 seconds.

import time

# Create a large collection
n = 1_000_000
big_list = list(range(n))
big_set = set(range(n))
target = n - 1   # worst case for list

# Time list lookup
start = time.perf_counter()
_ = target in big_list
list_time = time.perf_counter() - start

# Time set lookup
start = time.perf_counter()
_ = target in big_set
set_time = time.perf_counter() - start

print(f"List lookup: {list_time:.6f}s")
print(f"Set lookup:  {set_time:.6f}s")
print(f"Set is {list_time / max(set_time, 1e-9):.0f}x faster")

# You Should See:
# List lookup: ~0.01-0.02s
# Set lookup:  ~0.000001s
# Set is ~10000x faster (numbers vary by machine)

## Recap

| Structure | Key Pattern |
|---|---|
| List | `[x for x in data if condition]` — comprehensions are idiomatic |
| Dict | `.get(key, default)` — never crash on missing keys |
| Set | `set_a & set_b` — intersection for data reconciliation |
| Tuple | `a, b, c = tuple` — unpacking for multiple return values |
| Dataclass | `@dataclass` — replace boilerplate classes |
| List of dicts | The universal data exchange format in Python |

**Next notebook:** [Functions, Classes, and Lambdas](Python_Functions_Classes.ipynb)

**Full explanation:** [Data Structures Chapter](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/python/02_Data_Structures.md)